# Exploración — Resultados internacionales de fútbol

Se explora el dataset local `data/results.csv`. El objetivo posterior será predecir el resultado antes del partido.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
DATA_PATH = Path('data/results.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/results.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f'Dataset: {DATA_PATH.resolve()}')
print(f'Filas: {len(df):,} | Columnas: {df.shape[1]}')
df.head()

In [ ]:
required = {'date','home_team','away_team','home_score','away_score','tournament','city','country','neutral'}
assert not required - set(df.columns), f'Faltan: {required - set(df.columns)}'
before = len(df)
df = df.drop_duplicates().dropna(subset=list(required)).copy()
df['home_score'] = pd.to_numeric(df['home_score'], errors='raise')
df['away_score'] = pd.to_numeric(df['away_score'], errors='raise')
df['neutral'] = df['neutral'].astype(str).str.upper().eq('TRUE')
df = df.sort_values('date').reset_index(drop=True)
print(f'Rango: {df.date.min().date()} a {df.date.max().date()}')
print(f'Duplicados/nulos eliminados: {before - len(df):,}')
display(df.isna().sum().to_frame('nulos'))
df.describe(include='all')

In [ ]:
# El objetivo se obtiene de los goles. Los goles NO serán variables predictoras.
df['resultado'] = np.select(
    [df.home_score > df.away_score, df.home_score < df.away_score],
    ['gana_local', 'gana_visita'], default='empate'
)
df['year'] = df.date.dt.year
df['month'] = df.date.dt.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, x='resultado', order=['gana_local','empate','gana_visita'], ax=axes[0])
axes[0].set_title('Distribución del resultado')
axes[0].set_xlabel('')
df.groupby('year').size().tail(40).plot(ax=axes[1], title='Partidos por año')
axes[1].set_ylabel('Partidos')
plt.tight_layout()
plt.show()

print(f'Equipos únicos: {pd.concat([df.home_team, df.away_team]).nunique()}')
print(f'Torneos únicos: {df.tournament.nunique()}')
print(f'Sede neutral: {df.neutral.mean():.1%}')

In [ ]:
top = df.tournament.value_counts().head(12)
ax = sns.barplot(x=top.values, y=top.index)
ax.set(title='12 torneos con más partidos', xlabel='Partidos', ylabel='Torneo')
plt.tight_layout()
plt.show()

## Decisiones para el modelado

Se usarán equipos, torneo, país/ciudad sede, sede neutral y fecha. Se excluyen `home_score` y `away_score` porque revelan el resultado. El test será cronológico: se entrena con el pasado y se evalúa en los partidos más recientes.